# Real-Time Face Mask Compliance Detection

To use the CDS6334 environment, run below conda command:

In [ ]:
conda env export -n CDS6334 > CDS6334.yml

In [1]:
# import libraries
# !pip install ultralytics tensorflow opencv-python-headless matplotlib seaborn scikit-learn pandas
import os
import shutil
import random
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Deep Learning Imports
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from ultralytics import YOLO

print("Libraries installed and imported successfully.")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\huit5\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Libraries installed and imported successfully.


## 1.0 Data Loading

### 1.1 Define Paths

In [7]:
DATASET_PATH = r"C:\Users\huit5\OneDrive\Documents\Y3S2\Visual Information Processing\Project\Real-Time-Face-Mask-Detection\Dataset"
class_dirs = ["with_mask", "without_mask", "mask_weared_incorrect"]
print("DATASET_PATH exists:", os.path.exists(DATASET_PATH))
for d in class_dirs:
    p = os.path.join(DATASET_PATH, d)
    print(d, "exists:", os.path.exists(p), "num_images:", len(glob(os.path.join(p, "*.*"))))

DATASET_PATH exists: True
with_mask exists: True num_images: 2994
without_mask exists: True num_images: 2994
mask_weared_incorrect exists: True num_images: 2994


### 1.2 Helper Function: Parse XML

In [8]:
def parse_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    filename = root.find('filename').text
    size = root.find('size')
    width = int(size.find('width').text)
    height = int(size.find('height').text)
    
    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        bndbox = obj.find('bndbox')
        xmin = int(bndbox.find('xmin').text)
        ymin = int(bndbox.find('ymin').text)
        xmax = int(bndbox.find('xmax').text)
        ymax = int(bndbox.find('ymax').text)
        objects.append({
            'name': name,
            'xmin': xmin, 'ymin': ymin,
            'xmax': xmax, 'ymax': ymax
        })
    
    return filename, width, height, objects

### 1.3 Visual Sanity Check

In [9]:
def visualize_sample():
    xml_files = glob(os.path.join(ANNOTATIONS_PATH, "*.xml"))
    if not xml_files:
        print("No XML files found! Check your path.")
        return

    sample_xml = random.choice(xml_files)
    filename, w, h, objects = parse_xml(sample_xml)
    img_path = os.path.join(IMAGES_PATH, filename)
    
    if not os.path.exists(img_path):
        print(f"Image not found: {img_path}")
        return

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    for obj in objects:
        cv2.rectangle(img, (obj['xmin'], obj['ymin']), (obj['xmax'], obj['ymax']), (0, 255, 0), 2)
        cv2.putText(img, obj['name'], (obj['xmin'], obj['ymin']-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Sample: {filename}")
    plt.show()

visualize_sample()

No XML files found! Check your path.


## 2.0 Data Preprocessing

### 2.1 Preprocessing for Model A (YOLO Format)

#### 2.1.1 Convert Labels

In [ ]:
# YOLO cannot read XML. It needs .txt files where coordinates are normalized (between 0 and 1).

#### 2.1.2 Directory Split

In [ ]:
# Creates the specific folder structure YOLO demands: datasets/images/train, 
# datasets/images/val, datasets/labels/train, etc.

#### 2.1.3 Move Files

In [ ]:
# Randomly moves 80% of images to train and 20% to val.

#### 2.1.4 Create YAML

In [ ]:
# Writes a simple text file (data.yaml) that tells YOLO: "Here are the train images, 
# and the 3 class names are with_mask, without_mask, mask_weared_incorrect."

### 2.2 Preprocessing for Model B (EfficientNet Format)

#### 2.2.1 Create Crop Directories

In [ ]:
# Creates 3 folders: crops/with_mask, crops/without_mask, crops/incorrect.

#### 2.2.2 Crop and Sort

In [ ]:
# Logic: It loops through every image, looks at the bounding box, cuts only the face out of the image, 
# and saves that tiny face image into the correct folder.

# Why: EfficientNet is a classifier. It needs to look at just the face to decide if the mask is 
# correct or not.

#### 2.2.3 Data Generators

In [ ]:
# Sets up ImageDataGenerator which automatically loads these cropped images in batches and applies 
# "Augmentation" (randomly rotating them slightly) to make the model smarter.

## 3.0 Model A: One-Stage Detector (YOLOv8n)

### 3.1 Initialize Model

In [ ]:
# Downloads the yolov8n.pt (Nano) weights. 
# This model already knows what a "person" looks like, giving you a head start (Transfer Learning).

### 3.2 Training

In [ ]:
# The core training loop.
# Details: You run model.train(). It tries to predict boxes, checks how wrong it is (Loss), 
# and updates itself. You run this for ~20-50 epochs.

### 3.3 Evaluation (YOLO)

In [ ]:
# Calculates the score. Generates the mAP (Mean Average Precision). 
# plot a Confusion Matrix here to see if it confuses "Incorrect" with "Correct".

## 4.0 Model B: Two-Stage Classifier (EfficientNet-B0)

### 4.1 Build Architecture

In [ ]:
# Modifies the standard EfficientNet.

# load EfficientNet but cut off the "Head" (the top layer). 
# You replace it with your own "3-Class Output Layer" so it predicts your specific mask classes instead of generic objects.

### 4.2 Compile Model

In [ ]:
# Configures the learning rules.

# Sets the optimizer to Adam (standard for vision) and loss to CategoricalCrossentropy 
# (standard for multi-class classification).

### 4.3 Training

In [ ]:
# Trains the classifier on the cropped faces from Step 2.2.

### 4.4 Evaluation (EfficientNet)

In [ ]:
# checks accuracy specifically on the cropped faces.

# Generates a classification report showing Precision, Recall, and F1-score for each of the 3 classes.

## 5.0 Comparative Analysis

### 5.1 Metrics Comparison

In [ ]:
# Puts the results side-by-side.

# create a table or bar chart comparing the Recall for 'Incorrect Mask'. 
# Question answered: Which model is less likely to miss a bad mask?

### 5.2 Inference Speed Test (FPS)

In [ ]:
# Measures "Real-World" speed.

# Run YOLO on 100 images -> Measure time.
# Run Face Detect + EfficientNet on 100 images -> Measure time.
# Result: Show that YOLO is likely faster, but check if EfficientNet is "fast enough."

## 6.0 Application Simulation (Access Control)